In [52]:
import os
import pandas as pd
import commons as c
import numpy as np

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import plotly.graph_objects as go

# Boxplots of mutant detectability (distance between mutant and original)


In [53]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)
df_normal['nature'] = 'non-equivalent'

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)
df_equiv['nature'] = 'equivalent'
df = pd.concat([df_equiv, df_normal], ignore_index=True)

In [54]:
df.head()

,gates,depth,singlequbit_gates,multiqubit_gates,Input,Input_type,Algorithm,Qubits_number,Operator,Gate,...,threshold,metric,metric_full,ideal_label,noisy_label,ideal_distance,noisy_distance,correctness,hardware_named,nature
0,65,42,29,36,PureState_0,PureState,ae,8,Add,ch,...,A,H,Hellinger,True,False,0.066363,0.145299,False,Kyiv noise model,equivalent
1,65,42,29,36,Quratest_0,Quratest,ae,8,Add,ch,...,A,H,Hellinger,False,False,0.058578,0.114127,True,Kyiv noise model,equivalent
2,65,42,29,36,PureState_1,PureState,ae,8,Add,ch,...,A,H,Hellinger,False,False,0.059395,0.144436,True,Kyiv noise model,equivalent
3,65,42,29,36,Quratest_1,Quratest,ae,8,Add,ch,...,A,H,Hellinger,True,False,0.069584,0.126662,False,Kyiv noise model,equivalent
4,65,42,29,36,PureState_2,PureState,ae,8,Add,ch,...,A,H,Hellinger,True,False,0.065690,0.225888,False,Kyiv noise model,equivalent


# Best threshold: 

In [45]:
def find_best_thresholds(df, hw, m, start=0, end=1, steps=100, plot=True):
    thresholds = np.linspace(start, end, steps)
    f1_scores = []
    acc_scores = []

    best_thresh_f1 = None
    best_thresh_acc = None
    best_f1 = -1
    best_acc = -1

    # Convert 'equivalent' to 1 and 'non-equivalent' to 0
    y_true = df['nature'].map({'equivalent': 1, 'non-equivalent': 0}).values

    for t in thresholds:
        y_pred = (df['noisy_distance'] <= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        acc = accuracy_score(y_true, y_pred)
        f1_scores.append(f1)
        acc_scores.append(acc)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh_f1 = t
        if acc > best_acc:
            best_acc = acc
            best_thresh_acc = t

    if plot:
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=acc_scores, y=thresholds,
            mode='lines', name='Accuracy',
            line=dict(color='blue')
        ))

        fig.add_trace(go.Scatter(
            x=f1_scores, y=thresholds,
            mode='lines', name='F1 Score',
            line=dict(color='green')
        ))

        fig.add_hline(
            y=best_thresh_acc,
            line_dash="dash",
            line_color="blue",
            annotation_text=f"Best Acc @ {best_acc:.3f}",
            annotation_position="top left"
        )

        fig.add_hline(
            y=best_thresh_f1,
            line_dash="dash",
            line_color="green",
            annotation_text=f"Best F1 @ {best_f1:.3f}",
            annotation_position="top right"
        )
        
        output_folder = 'results/test_thresholds/'
        file_name = f"{hw}_acc_f1_{m}"
        c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[start, end], height=700)

    return best_thresh_f1, best_f1, best_thresh_acc, best_acc


In [46]:
hw = "kyiv"
metrics = {'H': (0.19,0.32), 'J': (0.25,0.275), 'T': (0.11,0.14), 'F': (0.97,1), 'E': (0.03,0.1)}
print(f"===================== {hw} =======================")
df_hw = df[df['hardware'] == hw]
for m, (start, end) in metrics.items():
    df_metric = df_hw[df_hw['metric'] == m]
    best_thresh_f1, best_f1, best_thresh_acc, best_acc = find_best_thresholds(df_metric, hw, m, start=start, end=end, steps=500)
    print(f"Metric: {m}, Best threshold: {best_thresh_acc}, Best Accuracy: {best_acc:.4f}")
    print(f"Metric: {m}, Best threshold: {best_thresh_f1}, Best F1 score: {best_f1:.4f}")

===================== kyiv =======================
Metric: H, Best threshold: 0.19521042084168336, Best Accuracy: 0.7303
Metric: H, Best threshold: 0.30827655310621244, Best F1 score: 0.7386
Metric: J, Best threshold: 0.2582665330661323, Best Accuracy: 0.7451
Metric: J, Best threshold: 0.26322645290581165, Best F1 score: 0.7647
Metric: T, Best threshold: 0.12412825651302606, Best Accuracy: 0.8696
Metric: T, Best threshold: 0.12418837675350702, Best F1 score: 0.8798
Metric: F, Best threshold: 0.9980160320641283, Best Accuracy: 0.4872
Metric: F, Best threshold: 0.9980160320641283, Best F1 score: 0.6552
Metric: E, Best threshold: 0.03336673346693387, Best Accuracy: 0.5790
Metric: E, Best threshold: 0.0939679358717435, Best F1 score: 0.6504


In [47]:
hw = "brisbane"
metrics = {'H': (0.22,0.28), 'J': (0.21,0.32), 'T': (0.8,0.12), 'F': (0.85,1), 'E': (0.03,0.3)}
print(f"===================== {hw} =======================")
df_hw = df[df['hardware'] == hw]
for m, (start, end) in metrics.items():
    df_metric = df_hw[df_hw['metric'] == m]
    best_thresh_f1, best_f1, best_thresh_acc, best_acc = find_best_thresholds(df_metric, hw, m, start=start, end=end, steps=300)
    print(f"Metric: {m}, Best threshold: {best_thresh_acc}, Best Accuracy: {best_acc:.4f}")
    print(f"Metric: {m}, Best threshold: {best_thresh_f1}, Best F1 score: {best_f1:.4f}")

===================== brisbane =======================
Metric: H, Best threshold: 0.23404682274247493, Best Accuracy: 0.7091
Metric: H, Best threshold: 0.2619397993311037, Best F1 score: 0.7041
Metric: J, Best threshold: 0.22103678929765885, Best Accuracy: 0.7229
Metric: J, Best threshold: 0.3111705685618729, Best F1 score: 0.7370
Metric: T, Best threshold: 0.12, Best Accuracy: 0.8772
Metric: T, Best threshold: 0.12, Best F1 score: 0.8880
Metric: F, Best threshold: 0.9974916387959867, Best Accuracy: 0.4872
Metric: F, Best threshold: 0.9974916387959867, Best F1 score: 0.6551
Metric: E, Best threshold: 0.03632107023411371, Best Accuracy: 0.5602
Metric: E, Best threshold: 0.3, Best F1 score: 0.6476


In [48]:
hw = "sherbrooke"
metrics = {'H': (0.24,0.35), 'J': (0.25,0.32), 'T': (0.1,0.2), 'F': (0.5,0.9), 'E': (0.03,0.3)}
print(f"===================== {hw} =======================")
df_hw = df[df['hardware'] == hw]
for m, (start, end) in metrics.items():
    df_metric = df_hw[df_hw['metric'] == m]
    best_thresh_f1, best_f1, best_thresh_acc, best_acc = find_best_thresholds(df_metric, hw, m, start=start, end=end, steps=300)
    print(f"Metric: {m}, Best threshold: {best_thresh_acc}, Best Accuracy: {best_acc:.4f}")
    print(f"Metric: {m}, Best threshold: {best_thresh_f1}, Best F1 score: {best_f1:.4f}")

===================== sherbrooke =======================
Metric: H, Best threshold: 0.2712709030100334, Best Accuracy: 0.6904
Metric: H, Best threshold: 0.34816053511705686, Best F1 score: 0.6968
Metric: J, Best threshold: 0.25210702341137126, Best Accuracy: 0.7027
Metric: J, Best threshold: 0.2977591973244147, Best F1 score: 0.7188
Metric: T, Best threshold: 0.1568561872909699, Best Accuracy: 0.8294
Metric: T, Best threshold: 0.1979933110367893, Best F1 score: 0.8427
Metric: F, Best threshold: 0.9, Best Accuracy: 0.3863
Metric: F, Best threshold: 0.9, Best F1 score: 0.5483
Metric: E, Best threshold: 0.04354515050167224, Best Accuracy: 0.5510
Metric: E, Best threshold: 0.3, Best F1 score: 0.6447


# Plot Acc/F1 as function of Threshold

In [63]:
def find_best_thresholds_2(df, hw, m, start=0, end=1, steps=100, plot=True):
    thresholds = np.linspace(start, end, steps)
    f1_scores = []
    acc_scores = []

    best_thresh_f1 = None
    best_thresh_acc = None
    best_f1 = -1
    best_acc = -1

    # Convert 'equivalent' to 1 and 'non-equivalent' to 0
    y_true = df['nature'].map({'equivalent': 1, 'non-equivalent': 0}).values

    for t in thresholds:
        y_pred = (df['noisy_distance'] <= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        acc = accuracy_score(y_true, y_pred)
        f1_scores.append(f1)
        acc_scores.append(acc)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh_f1 = t
        if acc > best_acc:
            best_acc = acc
            best_thresh_acc = t

    if plot:
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=thresholds, y=acc_scores,
            mode='lines', name='Accuracy',
            line=dict(color='blue')
        ))

        fig.add_trace(go.Scatter(
            x=thresholds, y=f1_scores,
            mode='lines', name='F1 Score',
            line=dict(color='green')
        ))

        fig.add_vline(
            x=best_thresh_acc,
            line_dash="dash",
            line_color="blue",
            annotation_text=f"Best Acc: {best_thresh_acc:.3f}",
            annotation_position="top left"
        )

        fig.add_vline(
            x=best_thresh_f1,
            line_dash="dash",
            line_color="green",
            annotation_text=f"Best F1: {best_thresh_f1:.3f}",
            annotation_position="top right"
        )

        fig.update_layout(
            xaxis_title="Threshold",
            yaxis_title="Score",
        )

        output_folder = 'results/test_thresholds/'
        file_name = f"{hw}_acc_f1_{m}_500_steps"
        c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1], height=700)

    return best_thresh_f1, best_f1, best_thresh_acc, best_acc


In [64]:
for hw in c.hardware:
    print(f"===================== {hw} =======================")
    df_hw = df[df['hardware'] == hw]
    for m in c.metrics:
        df_metric = df_hw[df_hw['metric'] == m]
        best_thresh_f1, best_f1, best_thresh_acc, best_acc = find_best_thresholds_2(df_metric, hw, m, steps=500)

===================== kyiv =======================
===================== brisbane =======================
===================== sherbrooke =======================
